# Fix Billex Headword — Batch

Post-processor for every `*_Billex.csv` produced by notebook 14.

**Problem.** Two OCR/typography artefacts contaminate the Indonesian side
of Billex entries across many dictionaries:

**(A) Trailing phonetic respelling.** KBBI-style dictionaries print
`"berak [bérak]"` where the bracketed form is the pronunciation guide. The
PDF extractor OCRs `[` as `i`/`(` and `]` as `l`/`)`, yielding headwords
like `"berak ibérak"`, `"agen iagén"`, `"demonstrasi idémonstrasil"`.

**(B) Syllable-split headwords.** The PDF writes `"ak.sa.ra"`. After dot
removal, stray spaces survive between syllables: `"aksa ra"`, `"abso lut"`,
`"bangsa wan"`.

Both patterns are **typographic features of the Indonesian side** (KBBI /
Balai Bahasa convention), so they apply uniformly to whichever column is
Indonesian. `LookupIsFromIndonesia.csv` tells us which column that is.

**Scope.** Regional-language columns are left untouched. Regional
orthographies across 32 dictionaries vary too much for a safe generic
cleaner — what looks like a "phonetic tail" in Jambi might be a legitimate
compound or reduplication in Madurese or Balinese.

Per dictionary you get two files:
- `<id>_Billex.csv` — pipeline-compatible, same schema as before
- `<id>_Billex_audit.csv` — adds `_original`, `_cleaned`, `cleanup_tag`,
  `direction` for evaluation/research use.

Plus two cross-dictionary files:
- `_summary.csv` — per-dictionary counts of each cleanup action
- `_review_multiword.csv` — all multi-word headwords the cleaner couldn't
  confidently fix, pooled across dictionaries (systematic extraction-
  boundary errors surface here)

## 1. Imports & paths

In [1]:
import re
import unicodedata
from pathlib import Path
from typing import Optional

import pandas as pd
from rapidfuzz import fuzz

from _common import (
    parse_dict_id,
    load_direction_lookup,
    direction_for,
    roles_for_billex,
)

SRC_DIR = Path("../Ekstraksi/9. Bilingual Lexicon")
DST_DIR = Path("../Ekstraksi/9. Bilingual Lexicon - Fixed")

# Tunable thresholds
PHONETIC_SIMILARITY_MIN = 80
SYLLABLE_MAX_COMBINED_LEN = 12
SYLLABLE_MIN_TOKEN_LEN = 2

DIACRITIC_RE = re.compile(r"[À-ÿ]")

assert SRC_DIR.exists(), f"Source dir not found: {SRC_DIR.resolve()}"
DST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading from: {SRC_DIR.resolve()}")
print(f"Writing to:   {DST_DIR.resolve()}")

Reading from: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\9. Bilingual Lexicon
Writing to:   C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\9. Bilingual Lexicon - Fixed


## 2. Pattern detectors

### Diacritic-insensitive comparison
`"bérak"` should match `"berak"` when deciding whether a tail is a
phonetic respelling of the head.

In [2]:
def strip_diacritics(s: str) -> str:
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


# Quick check
print(strip_diacritics("bérak"), "==", strip_diacritics("berak"))

berak == berak


### Pattern A — phonetic tail

Strips off a trailing token that looks like a bracket-wrapped phonetic
respelling of the head. OCR readings of `[` include `i`, `(`, `l`; of `]`
include `l`, `)`.

In [3]:
def looks_like_phonetic_tail(head: str, tail: str) -> bool:
    if not head or not tail:
        return False
    core = tail
    if core[0] in "i(l[":
        core = core[1:]
    if core and core[-1] in "l)]":
        core = core[:-1]
    if len(core) < 2:
        return False
    if abs(len(core) - len(head)) > 2:
        return False
    score = fuzz.ratio(strip_diacritics(head).lower(), strip_diacritics(core).lower())
    return score >= PHONETIC_SIMILARITY_MIN


for h, t in [("berak", "ibérak"), ("agen", "iagén"), ("bentuk", "iabénthukl"),
             ("cermin", "icermml"), ("berak", "something")]:
    print(f"  head={h!r:<10} tail={t!r:<15} -> {looks_like_phonetic_tail(h, t)}")

  head='berak'    tail='ibérak'        -> True
  head='agen'     tail='iagén'         -> True
  head='bentuk'   tail='iabénthukl'    -> True
  head='cermin'   tail='icermml'       -> False
  head='berak'    tail='something'     -> False


### Pattern B — syllable split

Rejoins two short alphabetic tokens that together form a plausibly single
Indonesian lemma.

In [4]:
def looks_like_syllable_split(a: str, b: str) -> bool:
    if not (a and b):
        return False
    if not (a.isalpha() and b.isalpha()):
        return False
    if DIACRITIC_RE.search(b):
        return False
    if len(a) + len(b) > SYLLABLE_MAX_COMBINED_LEN:
        return False
    if len(a) < SYLLABLE_MIN_TOKEN_LEN or len(b) < SYLLABLE_MIN_TOKEN_LEN:
        return False
    return True


for a, b in [("aksa", "ra"), ("abso", "lut"), ("bangsa", "wan"),
             ("supercalifragi", "listic"), ("a", "ba")]:
    print(f"  {a!r:<16} + {b!r:<6} -> {looks_like_syllable_split(a, b)}")

  'aksa'           + 'ra'   -> True
  'abso'           + 'lut'  -> True
  'bangsa'         + 'wan'  -> True
  'supercalifragi' + 'listic' -> False
  'a'              + 'ba'   -> False


## 3. Clean one headword

In [5]:
def clean_headword(raw: str) -> tuple[str, str]:
    """
    Returns (cleaned_headword, cleanup_tag).
    Tags: 'ok', 'phonetic_stripped', 'syllable_rejoined',
          'phonetic_stripped+syllable_rejoined', 'review_multiword'.
    """
    if not isinstance(raw, str):
        return raw, "ok"
    raw = raw.strip()
    if not raw:
        return raw, "ok"
    tokens = raw.split()
    if len(tokens) == 1:
        return raw, "ok"

    # Pattern A on last token
    if len(tokens) >= 2 and looks_like_phonetic_tail(tokens[-2], tokens[-1]):
        cleaned = " ".join(tokens[:-1])
        inner, inner_tag = clean_headword(cleaned)
        if inner_tag == "syllable_rejoined":
            return inner, "phonetic_stripped+syllable_rejoined"
        return inner, "phonetic_stripped"

    # Three tokens: first two are syllable-split of headword, third is phonetic
    if len(tokens) == 3 and looks_like_syllable_split(tokens[0], tokens[1]):
        joined = tokens[0] + tokens[1]
        if looks_like_phonetic_tail(joined, tokens[2]):
            return joined, "phonetic_stripped+syllable_rejoined"

    # Two tokens: syllable split
    if len(tokens) == 2 and looks_like_syllable_split(tokens[0], tokens[1]):
        return tokens[0] + tokens[1], "syllable_rejoined"

    return raw, "review_multiword"


# Quick test
for t in ["berak ibérak", "aksa ra", "aero bik iaerobik", "lalang lintang",
          "demonstrasi idémonstrasil"]:
    print(f"  {t!r:<30} -> {clean_headword(t)}")

  'berak ibérak'                 -> ('berak', 'phonetic_stripped')
  'aksa ra'                      -> ('aksara', 'syllable_rejoined')
  'aero bik iaerobik'            -> ('aerobik', 'phonetic_stripped+syllable_rejoined')
  'lalang lintang'               -> ('lalang lintang', 'review_multiword')
  'demonstrasi idémonstrasil'    -> ('demonstrasi', 'phonetic_stripped')


## 4. Process each dictionary

In [6]:
def process_one(
    csv_path: Path,
    direction_lookup: dict[str, int],
) -> tuple[Optional[dict], list[dict]]:
    dict_id = parse_dict_id(csv_path.name)
    if dict_id is None:
        return None, []

    direction = direction_for(dict_id, direction_lookup)
    if direction is None:
        direction = 1
        direction_known = False
    else:
        direction_known = True

    ind_col, reg_col = roles_for_billex(direction)

    df = pd.read_csv(csv_path)
    if ind_col not in df.columns:
        return None, []

    original = df[ind_col].astype(str).tolist()
    cleaned_with_tags = [clean_headword(v) for v in original]
    cleaned = [c[0] for c in cleaned_with_tags]
    tags = [c[1] for c in cleaned_with_tags]

    compat = df.copy()
    compat[ind_col] = cleaned

    audit = df.copy()
    audit[f"{ind_col}_original"] = original
    audit[f"{ind_col}_cleaned"] = cleaned
    audit["cleanup_tag"] = tags
    audit["direction"] = direction
    audit["direction_known"] = direction_known

    compat.to_csv(DST_DIR / f"{dict_id}_Billex.csv", index=False)
    audit.to_csv(DST_DIR / f"{dict_id}_Billex_audit.csv", index=False)

    tag_counts = pd.Series(tags).value_counts().to_dict()
    summary = {
        "dict_id": dict_id,
        "direction": direction,
        "direction_known": direction_known,
        "indonesian_column": ind_col,
        "rows": len(df),
        "unchanged": tag_counts.get("ok", 0),
        "phonetic_stripped": (
            tag_counts.get("phonetic_stripped", 0)
            + tag_counts.get("phonetic_stripped+syllable_rejoined", 0)
        ),
        "syllable_rejoined": (
            tag_counts.get("syllable_rejoined", 0)
            + tag_counts.get("phonetic_stripped+syllable_rejoined", 0)
        ),
        "review_multiword": tag_counts.get("review_multiword", 0),
    }

    review_rows = []
    for orig, clean, tag in zip(original, cleaned, tags):
        if tag == "review_multiword":
            review_rows.append({
                "dict_id": dict_id,
                "indonesian_column": ind_col,
                "original": orig,
            })

    return summary, review_rows

In [7]:
direction_lookup = load_direction_lookup()
all_csvs = sorted(
    p for p in SRC_DIR.glob("*_Billex.csv") if not p.name.startswith(".")
)
print(f"Found {len(all_csvs)} Billex CSVs")

summaries, all_review_rows = [], []
for csv in all_csvs:
    summary, review_rows = process_one(csv, direction_lookup)
    if summary is not None:
        summaries.append(summary)
        all_review_rows.extend(review_rows)

summary_df = pd.DataFrame(summaries).sort_values(
    "dict_id", key=lambda s: s.astype(int)
)
summary_df.to_csv(DST_DIR / "_summary.csv", index=False)

review_df = pd.DataFrame(all_review_rows)
review_df.to_csv(DST_DIR / "_review_multiword.csv", index=False)

summary_df

Found 68 Billex CSVs


,dict_id,direction,direction_known,indonesian_column,rows,unchanged,phonetic_stripped,syllable_rejoined,review_multiword
10,1,0,True,kata_tujuan,16,11,0,0,5
21,2,0,True,kata_tujuan,76,64,0,0,12
30,3,0,True,kata_tujuan,19,18,0,0,1
38,4,1,True,kata_asal,1894,1668,102,85,50
47,5,1,True,kata_asal,2395,1953,76,161,215
...,...,...,...,...,...,...,...,...,...
61,85,0,True,kata_tujuan,26,22,0,0,4
62,87,1,True,kata_asal,761,621,0,46,94
63,89,0,True,kata_tujuan,1128,922,0,0,206
65,91,0,True,kata_tujuan,10077,7968,14,0,2095


## 5. Totals

In [8]:
print(f"Dictionaries processed:     {len(summary_df)}")
print(f"Dictionaries w/o direction: {(~summary_df['direction_known']).sum()}")
print(f"Total entries:              {summary_df['rows'].sum()}")
print(f"Phonetic-tail stripped:     {summary_df['phonetic_stripped'].sum()}")
print(f"Syllable splits rejoined:   {summary_df['syllable_rejoined'].sum()}")
print(f"Review-multiword remaining: {summary_df['review_multiword'].sum()}")
print(f"\nOutputs written to: {DST_DIR.resolve()}")

Dictionaries processed:     68
Dictionaries w/o direction: 0
Total entries:              107732
Phonetic-tail stripped:     1491
Syllable splits rejoined:   3255
Review-multiword remaining: 19070

Outputs written to: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\9. Bilingual Lexicon - Fixed


## 6. Review multi-word headwords

These are entries the cleaner was not confident enough to auto-fix. They
tend to be upstream extraction-boundary errors where one entry's content
bled into the next — a distinct error class from spelling.

In [9]:
if len(review_df) > 0:
    print(f"{len(review_df)} multi-word entries flagged for review across "
          f"{review_df['dict_id'].nunique()} dictionaries.\n")
    print("Top 20 examples:")
    print(review_df.head(20).to_string(index=False))
else:
    print("No multi-word entries needed review.")

19070 multi-word entries flagged for review across 58 dictionaries.

Top 20 examples:
dict_id indonesian_column                            original
     10         kata_asal                          me ng ajar
     10         kata_asal                           b as ko m
     10         kata_asal                        berba ta san
     10         kata_asal                            ber ba u
     10         kata_asal                          membi bi t
     10         kata_asal                             b il as
     10         kata_asal                      mencanti kk an
     10         kata_asal                          mcnca ta t
     10         kata_asal                      cepat-cepat ad
     10         kata_asal                             cukur v
     10         kata_asal                          pencu k ur
     10         kata_asal                         k ec ul asa
     10         kata_asal              cungkil  icnc ung ki l
     10         kata_asal                     